In [2]:
%pip install pandas xgboost shap scikit-learn flask matplotlib seaborn


  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached numpy-2.0.2-cp312-cp312-win_amd64.whl.metadata (59 kB)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 1.1 MB/s eta 0:01:31
   ---------------------------------------- 0.8/101.7 MB 1.0 MB/s eta 0:01:40
   ---------------------------------------- 0.8/101.7 MB 1.0 MB/s eta 0:01:40
   ---------------------------------------- 1.0/101.7 MB 825.2 kB/s eta 0:02:02
   ---------------------------------------- 1.0/101.7 MB 825.2 kB/s eta 0:02:02
    --------------------------------------- 1.3/101.7 MB 789.6 kB/s eta 0:02:08
    --------------------------------------- 1.6/101.7 MB 806.6 kB/s eta 0:02:05
    -------------------------

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.


In [3]:
import pandas as pd
import xgboost as xgb
import shap
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [4]:
# 1. Load Dataset
df = pd.read_csv('../data/credit_risk_dataset.csv')

In [5]:
# 2. Basic Cleaning
# Handling missing values (Common in credit data: employment length and interest rate)
df['person_emp_length'].fillna(df['person_emp_length'].median(), inplace=True)
df['loan_int_rate'].fillna(df['loan_int_rate'].median(), inplace=True)

C:\Users\mksme\AppData\Local\Temp\ipykernel_5368\1066346776.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['person_emp_length'].fillna(df['person_emp_length'].median(), inplace=True)
C:\Users\mksme\AppData\Local\Temp\ipykernel_5368\1066346776.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

In [6]:
# 3. Encoding Categorical Data
# Converting things like 'Home Ownership' and 'Loan Intent' to numbers
le = LabelEncoder()
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

In [7]:
# 4. Split Data
X = df.drop('loan_status', axis=1)
y = df['loan_status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# 5. Train XGBoost Model
# XGBoost is great for tabular credit data
model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1)
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [9]:
# 6. SHAP Explainer (The "Explainable AI" part)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

In [10]:
# 7. Save the Model and Explainer for our Flask App
with open('../models/model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('../models/explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)

print("Model and Explainer saved successfully!")

Model and Explainer saved successfully!
